## Amylase Tutorial

This tutorial showcases how *biocentral* can be used to analyze a dataset and train a model.

Required file: `amylase_pet.fasta`: Fasta file with dataset - Amylase PET: Single and double mutations of Amylase
(PDB: 1UA7) with normalized expression levels as targets - Source:
https://github.com/the-protein-engineering-tournament/pet-pilot-2023/tree/main/in_vitro

### Create the biocentral object

First, we create the biocentral object. We can choose between local and api mode. In local mode, biocentral will use the local installation of biotrainer. In api mode, biocentral will use the biotrainer API. We can also specify a custom API endpoint (e.g. to target only locally running servers).

In [1]:
from biocentral import Biocentral, BiocentralChart
from biocentral_api import BiocentralAPI
from biotrainer_core.input_files import read_FASTA

#biocentral = Biocentral(mode="local")
biocentral = Biocentral(mode="api", custom_api=BiocentralAPI(local_only=True))

/home/sebie/IdeaProjects/biocentral/biocentral_python/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found healthy biocentral servers at:
  http://localhost:9540 - v2.0.0


### Load and analyze the dataset

Next, we load the dataset and analyze the label distribution. The `BiocentralChart` class automatically determines the correct visualization scheme given the kind of labels (class/regression).

As this is fitness data, most fitness values are closed to the wildtype at zero. We also see some heavy outliers at >=10.

In [2]:
amylase_data = read_FASTA("amylase_pet.fasta")

label_chart = BiocentralChart.label_distribution(amylase_data)
label_chart.chart.interactive()

alt.Chart(...)

## Visualize the dataset in space

Next, we can visualize the dataset using an embedding method. Here, we keep it simple and use `one_hot_encoding` with UMAP 2D. We observe some mild clustering of the data. *Note that this functionality is currently only available via the 'api' mode.*

In [6]:
projection_config = {"n_components": "2"}

projection_result = biocentral.project(embedder_name="one_hot_encoding", method="umap", sequence_data=amylase_data, projection_config=projection_config)
projection_chart = BiocentralChart.projection_result(projection_result=projection_result, dataset=amylase_data)
projection_chart.chart.interactive()

alt.Chart(...)

## Train a model

After analyzing the dataset and doing an unsupervised clustering, we now want to train a **supervised model** to predict the fitness data of unknown mutations.

In [8]:
model_config = {"model_choice": "FNN", "protocol": "sequence_to_value", "embedder_name": "one_hot_encoding"}

model_result = biocentral.train(config=model_config, training_data=amylase_data)
print(model_result)

config={'device': 'cuda', 'seed': 42, 'save_split_ids': False, 'sanity_check': True, 'ignore_file_inconsistencies': False, 'external_writer': 'none', 'output_dir': '/tmp/tmpis77pip8', 'bootstrapping_iterations': 30, 'force_execution': False, 'model_choice': 'FNN', 'optimizer_choice': 'adam', 'learning_rate': 0.001, 'dropout_rate': 0.25, 'epsilon': 0.001, 'loss_choice': 'mean_squared_error', 'disable_pytorch_compile': False, 'num_epochs': 200, 'batch_size': 128, 'patience': 10, 'shuffle': True, 'use_class_weights': False, 'auto_resume': False, 'limited_sample_size': -1, 'embedder_name': 'one_hot_encoding', 'use_half_precision': False, 'cross_validation_config': {'method': 'hold_out', 'choose_by': 'loss'}, 'protocol': 'sequence_to_value', 'input_data': 3924, 'log_dir': '/tmp/tmpis77pip8/12d7bcd62094f6c9'} derived_values=DerivedValues(biotrainer_version='2.0.0', class_int2str=None, class_str2int=None, computed_class_weights=None, embedding_stats=None, embeddings_file=None, model_hash='12d

## Model Results

Let's look at the basic model results: The training/validation loss curves shows the evolution of the loss during training. We observe almost no decrease in loss over time, indicating that the model is barely learning from the data. The **test set performance** chart shows the performance of the model on the test set. We observe a Spearman's correlation coefficient of around 0.17, which is only a little bit better than random performance.

In [9]:
loss_curve_chart = BiocentralChart.model_loss_curve(model_result)
loss_curve_chart.chart.interactive()

alt.Chart(...)

In [12]:
test_set_performance = BiocentralChart.model_test_set_performance(model_result, metric_name="spearmans-corr-coeff")
test_set_performance.chart.interactive()

alt.Chart(...)

## Inference

Let's use our trained model to create new predictions for unseen mutations.

In [13]:
inference_result = biocentral.inference(model_result, inference_data={"Mut0": "LMAPSIKSGTILHAWNWSFNTLKHNMKDIHDAGYTAIQTSPINQVKEGNQGDKSMSNWYWLYQPTSYQIGNRYLGTEQEFKEMCAAAEEYGIKVIVDAVINHTTSDYAAISNEVKSIPNWTHGNTPIKNWSDRWDVTQNSLSGLYDWNTQNTQVQSPLKRFLDRALNDGADGFRFDAAKHIELPDDGSYGSQFWPNITNTSAEFQYGEILQDSVSRDAAYANYMDVTASNYGHSIRSALKNRNLGVSNISHYAVDVSADKLVTWVESHDTYANDDEESTWMSDDDIRLGKAVIASRSGSTPLFFSRPEGGGNGVRFPGKSQIGDRGSALFEDQAITAVNRFHNVMAGQPEELSNPNGNNQIFMNQRGSHGVVLANAGSSSVSINTATKLPDGRYDNKAGAKSFQVNDGKLTGTINARSVAVLYPX"})
print(inference_result)

predictions=[BiotrainerPrediction(seq_id='Mut0', prediction=Prediction1(anyof_schema_1_validator=None, anyof_schema_2_validator=0.0010662749409675598, anyof_schema_3_validator=None, actual_instance=0.0010662749409675598, any_of_schemas={'float', 'List[object]', 'str'}), is_aggregated=False, residue_index=None, raw_prediction=RawPrediction(anyof_schema_1_validator=None, anyof_schema_2_validator=0.0010662749409675598, anyof_schema_3_validator=None, actual_instance=0.0010662749409675598, any_of_schemas={'float', 'List[object]', 'str'}), mcd_predictions=None, mcd_mean=None, mcd_std=None, mcd_lower_bound=None, mcd_upper_bound=None, bald_score=None)] metrics=None


## Wrap Up

You completed the amylase tutorial for *biocentral*. Thank you for using *biocentral*! Of course, there is much to improve and a lot more to explore - our `one_hot_encoding` model did not perform well - maybe *protein language models* such as ProtT5 or ESM-2 can perform better? Feel free to try them out as a next step!